In [6]:
import os
import json
import numpy as np
from scipy.stats import t, norm

In [ ]:
def calcular_kpis_promedio_con_ic_en_formato_json(directorio, nivel_confianza=95, guardar_en=None):
    """
    Calcula el promedio ± IC de todos los KPIs numéricos en archivos JSON de un directorio.
    Usa t-Student si n < 100, o normal estándar (z) si n >= 100.

    Args:
        directorio (str): Ruta a carpeta con archivos JSON.
        nivel_confianza (int or float): Nivel de confianza deseado (ej. 90, 95, 99).
        guardar_en (str or None): Ruta para guardar el resultado en un JSON (opcional).

    Returns:
        dict: Diccionario anidado con 'media ± error' y metadata de cálculo.
    """
    if not (50 <= nivel_confianza < 100):
        raise ValueError("El nivel de confianza debe estar entre 50 y 99.9")

    prob = nivel_confianza / 100
    json_files = [os.path.join(directorio, f) for f in os.listdir(directorio) if f.endswith(".json")]
    n = len(json_files)
    if n == 0:
        raise ValueError("No se encontraron archivos JSON en el directorio.")

    # Usar el primer archivo como referencia de estructura
    with open(json_files[0], "r") as f:
        ejemplo = json.load(f)

    def extraer_rutas(d, prefijo=""):
        rutas = []
        for k, v in d.items():
            ruta = f"{prefijo}.{k}" if prefijo else k
            if isinstance(v, dict):
                rutas += extraer_rutas(v, ruta)
            elif isinstance(v, (int, float)):
                rutas.append(ruta)
        return rutas

    kpis_ruta = extraer_rutas(ejemplo)

    # Recolectar valores de cada KPI
    data = {kpi: [] for kpi in kpis_ruta}
    for file in json_files:
        with open(file, "r") as f:
            contenido = json.load(f)
            for kpi in kpis_ruta:
                try:
                    val = contenido
                    for key in kpi.split("."):
                        val = val[key]
                    if isinstance(val, (int, float)):
                        data[kpi].append(val)
                except (KeyError, TypeError):
                    continue

    resumen = {"_info": {"n": n, "nivel_confianza": f"{nivel_confianza}%", "distribucion": ""}}

    if n >= 100:
        z_val = norm.ppf((1 + prob) / 2)
        resumen["_info"]["distribucion"] = "normal"
    else:
        resumen["_info"]["distribucion"] = "t_student"

    for kpi, valores in data.items():
        arr = np.array(valores)
        mean = np.mean(arr)
        std = np.std(arr, ddof=1)
        sem = std / np.sqrt(n)
        if n >= 100:
            error = z_val * sem
        else:
            t_val = t.ppf((1 + prob) / 2, df=n - 1)
            error = t_val * sem
        valor_str = f"{round(mean, 2)} ± {round(error, 2)}"

        puntero = resumen
        keys = kpi.split(".")
        for key in keys[:-1]:
            puntero = puntero.setdefault(key, {})
        puntero[keys[-1]] = valor_str

    if guardar_en:
        with open(guardar_en, "w") as f:
            json.dump(resumen, f, indent=4)

    return resumen

In [15]:
resumen = calcular_kpis_promedio_con_ic_en_formato_json("resultados simulacion/ModeloA_None_T4500_C4208/kpis", nivel_confianza=95, guardar_en=None)
display(resumen)

{'_info': {'n': 40, 'nivel_confianza': '95%', 'distribucion': 't_student'},
 'LOS_hospitalizado': {'por_hospital_y_unidad': {'Hospital_1': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': '71.38 ± 0.22',
    'OR': '13.51 ± 0.01',
    'SDU_WARD': '169.74 ± 0.19'},
   'Hospital_2': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': '71.48 ± 0.18',
    'OR': '13.45 ± 0.01',
    'SDU_WARD': '162.66 ± 0.2'},
   'Hospital_3': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': '67.03 ± 0.18',
    'OR': '13.24 ± 0.01',
    'SDU_WARD': '159.97 ± 0.15'}},
  'promedio_por_hospital': {'Hospital_1': '225.59 ± 0.38',
   'Hospital_2': '222.48 ± 0.37',
   'Hospital_3': '222.76 ± 0.26'},
  'promedio_por_unidad': {'ED': '0.0 ± 0.0',
   'GA': '0.0 ± 0.0',
   'ICU': '69.96 ± 0.12',
   'OR': '13.4 ± 0.01',
   'SDU_WARD': '164.06 ± 0.1'}},
 'LOS_lista_espera': '11.95 ± 0.86',
 'costo_diario_promedio': {'social': '199.09 ± 7.83',
  'derivaciones_wl': '0.0 ± 0.0',
  'derivaciones_ed': '858.45 ± 24.